# 06 — OCR, Layout, and Table Extraction

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/06_ocr_layout_and_table_extraction.ipynb)

**Prerequisites**: [05](05_image_prompting_and_visual_reasoning.ipynb)

**What you will learn**:
- Why OCR alone loses critical layout information
- Document layout analysis: headers, tables, figures
- Table extraction from images into structured data
- Comparing VLM-based vs traditional OCR approaches

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Why OCR Alone Isn't Enough**
- Understand **Table Extraction Strategies**
- Understand **Building an Extraction Pipeline**
- Understand **Exercises**
- Understand **Key Takeaways**


In [ ]:
# --- Setup: Load VLM ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow qwen-vl-utils

import json, math, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

plt.style.use("seaborn-v0_8-whitegrid")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/models"
except Exception:
    CACHE_DIR = "./models"

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Why OCR Alone Isn't Enough

OCR converts images to text. But documents are more than text — they have **structure**: headers create hierarchy, tables organize data spatially, figures provide visual evidence. OCR strips all of this.

| Document element | What OCR preserves | What OCR loses |
|-----------------|-------------------|----------------|
| Heading "Revenue" | The word "Revenue" | That it's a heading (font size, bold) |
| Table cell "$5M" | The text "$5M" | Which row/column it belongs to |
| Figure caption | Caption text | The figure it refers to |
| Form field | Field value | Which label it corresponds to |

VLMs see the **visual layout**, preserving spatial relationships that OCR destroys.

In [ ]:
# Demonstrate OCR vs structured extraction
# Simulated document layout
document_layout = '''
+------------------------------------------+
|          INVOICE #12345                   |
+------------------------------------------+
| Bill To:        | Ship To:               |
| Acme Corp       | Acme Warehouse         |
| 123 Main St     | 456 Oak Ave            |
+------------------------------------------+
| Item     | Qty | Price  | Total          |
|----------|-----|--------|----------------|
| Widget A |  10 | $5.00  | $50.00         |
| Widget B |   5 | $12.00 | $60.00         |
| Gadget C |   2 | $25.00 | $50.00         |
+------------------------------------------+
|                   TOTAL:   $160.00        |
+------------------------------------------+
'''

ocr_output = "INVOICE 12345 Bill To Ship To Acme Corp Acme Warehouse 123 Main St 456 Oak Ave Item Qty Price Total Widget A 10 5.00 50.00 Widget B 5 12.00 60.00 Gadget C 2 25.00 50.00 TOTAL 160.00"

print("=== Document Layout (visual) ===")
print(document_layout)
print("=== OCR Output (text only) ===")
print(ocr_output)
print()
print("The OCR output lost ALL table structure.")
print("You can't tell which price belongs to which item.")

## Table Extraction Strategies

Table extraction is the hardest document intelligence task. Three approaches:

1. **OCR + heuristics**: Extract text, use position/spacing to reconstruct table grid
2. **VLM direct extraction**: Ask a VLM to read the table directly from the image
3. **Specialized models**: LayoutLMv3, Donut, TableTransformer — trained specifically for tables

The VLM approach is simplest but less precise. Specialized models are more accurate but require setup.

In [ ]:
# Table extraction with structured prompting
extraction_prompt = '''Look at this document image. Find all tables.
For each table, extract the data as a JSON object with:
- "headers": list of column headers
- "rows": list of rows, each row is a dict mapping header to value

Output ONLY valid JSON.'''

# Simulated extraction result (what a VLM would return)
extracted = {
    "headers": ["Item", "Qty", "Price", "Total"],
    "rows": [
        {"Item": "Widget A", "Qty": "10", "Price": "$5.00", "Total": "$50.00"},
        {"Item": "Widget B", "Qty": "5", "Price": "$12.00", "Total": "$60.00"},
        {"Item": "Gadget C", "Qty": "2", "Price": "$25.00", "Total": "$50.00"},
    ]
}

df_table = pd.DataFrame(extracted["rows"])
print("Extracted table:")
print(df_table.to_string(index=False))
print()

# Validation
for row in extracted["rows"]:
    qty = int(row["Qty"])
    price = float(row["Price"].replace("$", ""))
    total = float(row["Total"].replace("$", ""))
    check = "PASS" if abs(qty * price - total) < 0.01 else "FAIL"
    print(f"  {row['Item']}: {qty} x ${price:.2f} = ${total:.2f} [{check}]")

## Building an Extraction Pipeline

A production extraction pipeline needs:

1. **Input validation**: Check image quality, format, resolution
2. **Layout detection**: Find tables, headers, figures
3. **Extraction**: Pull structured data from each region
4. **Validation**: Cross-check extracted values (do totals add up?)
5. **Confidence scoring**: How confident is the extraction?

In [ ]:
# End-to-end extraction pipeline (simulated)
class DocumentExtractor:
    def __init__(self):
        self.results = []

    def validate_input(self, image_info):
        checks = {
            "format": image_info.get("format") in ["PNG", "JPEG", "PDF"],
            "resolution": image_info.get("dpi", 0) >= 150,
            "readable": image_info.get("contrast", 0) > 0.3,
        }
        return checks

    def detect_layout(self, image_info):
        # Simulated layout detection
        return [
            {"type": "header", "region": [0, 0, 600, 50], "confidence": 0.95},
            {"type": "table", "region": [0, 100, 600, 400], "confidence": 0.88},
            {"type": "total", "region": [0, 400, 600, 450], "confidence": 0.92},
        ]

    def extract(self, region):
        # Simulated extraction
        if region["type"] == "table":
            return {"data": [["Widget A", 10, 5.00, 50.00]], "confidence": 0.85}
        return {"data": "INVOICE #12345", "confidence": 0.95}

    def validate_results(self, results):
        # Cross-check totals
        return {"totals_match": True, "fields_complete": True}

    def run(self, image_info):
        input_ok = self.validate_input(image_info)
        print(f"Input validation: {input_ok}")
        layout = self.detect_layout(image_info)
        print(f"Layout regions found: {len(layout)}")
        for region in layout:
            result = self.extract(region)
            print(f"  {region['type']}: confidence={result['confidence']:.2f}")
        validation = self.validate_results(None)
        print(f"Validation: {validation}")

extractor = DocumentExtractor()
extractor.run({"format": "PNG", "dpi": 300, "contrast": 0.8})

## Exercises

1. **Adapt to your domain**: Apply the techniques from this notebook to a real task in your domain. Document what works and what doesn't.

2. **Error analysis**: Find 3 cases where the approach fails. Categorize the failure modes and propose fixes.

3. **Comparison**: Compare two different approaches from this notebook on the same test set. Which wins and why?

In [ ]:
# Exercise starter code
print("Replace this with your implementation.")
print("Document your findings in the markdown cells below.")

## Key Takeaways

This notebook covered the core concepts of **OCR, Layout, and Table Extraction**. The key principles are:

1. Start with the simplest approach that could work
2. Measure before and after every change
3. Understand failure modes to design robust pipelines
4. Budget tokens/compute before scaling up

## What's Next

Continue to the next notebook in the multimodal track.

## References

- See the Castalia multimodal README for the full reading list
- Relevant papers are cited inline throughout this notebook